In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import tqdm
import gzip
import json

import matplotlib
import pickle

import torch

from sklearn.manifold import TSNE

from models import *
from neighbors import get_nearest_neighbors

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Update these
data_dir = "../../sepsis-ai-modeling/250125_condensed_variables"
model_dir = "../../sepsis-ai-modeling/250125_condensed_variables/feature_weighted_autoencoder"

In [ ]:
AS_IS_COLUMNS = []
LOG_NORM_COLUMNS = [
    "SpO2", "BUN", "Creatinine", "AST", "ALT", 
    "Total Bili", "Direct Bili", "INR", "Input Last 24 h", "Output Last 24 h",
    "Hypertonic Crystalloid", "Hypotonic Crystalloid", "Isotonic Crystalloid",
    "Blood Products", "Isotonic Colloid",
    "Dopamine", "Epinephrine", "Norepinephrine", "Vasopressin", "Phenylephrine"
]

COMORBIDITIES = [
    'congestive_heart_failure', 'cardiac_arrhythmias', 'valvular_disease',
    'pulmonary_circulation', 'peripheral_vascular', 'hypertension',
    'paralysis', 'other_neurological', 'chronic_pulmonary',
    'diabetes_uncomplicated', 'diabetes_complicated', 'hypothyroidism',
    'renal_failure', 'liver_disease', 'peptic_ulcer',
    'aids', 'lymphoma', 'metastatic_cancer',
    'solid_tumor', 'rheumatoid_arthritis', 'coagulopathy',
    'obesity', 'weight_loss', 'fluid_electrolyte',
    'blood_loss_anemia', 'deficiency_anemias', 'alcohol_abuse',
    'drug_abuse', 'psychoses', 'depression'
]

EXCLUDE_COLUMNS = []

def format_states(df, scaler=None, all_columns=None):
    global AS_IS_COLUMNS, LOG_NORM_COLUMNS, EXCLUDE_COLUMNS
    dummies = pd.get_dummies(df, 
                        columns=[c for c in df.columns 
                                if pd.api.types.is_object_dtype(df[c].dtype) 
                                or pd.api.types.is_string_dtype(df[c].dtype) 
                                or isinstance(df[c].dtype, pd.CategoricalDtype)])
    if all_columns is not None:
        dummies = dummies.assign(**{c: np.zeros(len(dummies), dtype=np.uint8)
                                    for c in all_columns if c not in dummies.columns})
    if scaler is None:
        EXCLUDE_COLUMNS = [c for c in dummies.columns
                           if (('O2 Delivery Device' in c or c.startswith("Heart Rhythm")) and dummies[c].mean() < 0.01) or
                           c.endswith("Present")]
        dummies = dummies.drop(columns=EXCLUDE_COLUMNS)
        AS_IS_COLUMNS = ["id", "intime", "Invasive Ventilation", "Non-invasive Ventilation",
                 *[c for c in dummies.columns 
                   if "_" in c or 
                   " Present" in c or 
                   c.endswith("Current") or 
                   c.endswith("Ever") or 
                   (c.startswith("Positive") and c.endswith("Culture")) or 
                   c in COMORBIDITIES]]
        
        scaler = DataNormalization(dummies,
                                      as_is_columns=AS_IS_COLUMNS,
                                      log_norm_columns=LOG_NORM_COLUMNS,
                                      norm_columns=[c for c in dummies.columns if c not in AS_IS_COLUMNS + LOG_NORM_COLUMNS])
        return scaler.transform(dummies).assign(id=dummies['id'], intime=dummies['intime']), scaler, dummies.columns
    return scaler.transform(dummies.drop(columns=[c for c in EXCLUDE_COLUMNS if c in dummies.columns])).assign(id=dummies['id'], intime=dummies['intime'])
    
train_df, normalizer, all_cols = format_states(pd.read_csv(os.path.join(data_dir, "model_features", "inputs_train.csv"), index_col=0))
train_df = train_df.clip(-10, 10).assign(id=train_df['id'], intime=train_df['intime'])
val_df = format_states(pd.read_csv(os.path.join(data_dir, "model_features", "inputs_val.csv"), index_col=0), scaler=normalizer, all_columns=all_cols)
val_df = val_df.clip(-10, 10).assign(id=val_df['id'], intime=val_df['intime'])
test_df = format_states(pd.read_csv(os.path.join(data_dir, "model_features", "inputs_test.csv"), index_col=0), scaler=normalizer, all_columns=all_cols)
test_df = test_df.clip(-10, 10).assign(id=test_df['id'], intime=test_df['intime'])
print("Excluding columns:", EXCLUDE_COLUMNS)

train_uo = pd.read_csv(os.path.join(data_dir, "filters", "uo", "train.csv"), index_col=0).values[:,-1]
val_uo = pd.read_csv(os.path.join(data_dir, "filters", "uo", "val.csv"), index_col=0).values[:,-1]
test_uo = pd.read_csv(os.path.join(data_dir, "filters", "uo", "test.csv"), index_col=0).values[:,-1]
train_cmo = pd.read_csv(os.path.join(data_dir, "filters", "cmo", "train.csv"), index_col=0).values[:,-1]
val_cmo = pd.read_csv(os.path.join(data_dir, "filters", "cmo", "val.csv"), index_col=0).values[:,-1]
test_cmo = pd.read_csv(os.path.join(data_dir, "filters", "cmo", "test.csv"), index_col=0).values[:,-1]

# filter inputs to exclude patients who have less than 1 urine observation per day on average
train_mask = ~train_uo.astype(bool) & ~train_cmo.astype(bool) # pd.isna(train_y).groupby(train_df['id']).transform("sum") < max_missing
val_mask = ~val_uo.astype(bool) & ~val_cmo.astype(bool) # pd.isna(val_y).groupby(val_df['id']).transform("sum") < max_missing
test_mask = ~test_uo.astype(bool) & ~test_cmo.astype(bool) # pd.isna(test_y).groupby(test_df['id']).transform("sum") < max_missing

# filter to a maximum of 14 days per patient (84 timesteps), minimum of 12 hours (3 timesteps)
max_steps = np.round(np.quantile(train_df.groupby('id').size(), 0.95))
print('max steps:', max_steps)
min_days = 0.5
time_res = 4
train_mask &= ((train_df['intime'].groupby(train_df['id']).transform(lambda g: np.argsort(g)) < max_steps)
              & (pd.Series(train_mask, index=train_df.index).groupby(train_df['id']).transform("sum") >= min_days * 24 / time_res))
val_mask &= ((val_df['intime'].groupby(val_df['id']).transform(lambda g: np.argsort(g)) < max_steps)
            & (pd.Series(val_mask, index=val_df.index).groupby(val_df['id']).transform("sum") >= min_days * 24 / time_res))
test_mask &= ((test_df['intime'].groupby(test_df['id']).transform(lambda g: np.argsort(g)) < max_steps)
             & (pd.Series(test_mask, index=test_df.index).groupby(test_df['id']).transform("sum") >= min_days * 24 / time_res))

# TEMPORARY: sample test IDs
# np.random.seed(1234)
# test_sample_ids = np.random.choice(test_df['id'].unique(), size=2000, replace=False)
# test_mask &= test_df['id'].isin(test_sample_ids)

print("Before:", len(train_df), len(val_df), len(test_df))
train_df = train_df[train_mask].reset_index(drop=True)
val_df = val_df[val_mask].reset_index(drop=True)
test_df = test_df[test_mask].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))

In [ ]:
best_param_set = {'architecture': 'transformer', 
                  'nencoder': 4, 
                  'nhead': 8, 
                  'nhid': 128, 
                  'nembed': 32, 
                  'lr': 0.001, 
                  'dropout': 0.1, 
                  'lr_decay': 0.98, 
                  'n_warmup': 4, 
                  'mask_gamma': 1.01}
model_trainer = TimeSeriesAutoencoderTrainer(
    train_df,
    val_df,
    test_df,
    time_col='intime',
    device='cpu',
    checkpoint_path=os.path.join(model_dir, f"weighted_transformer.pt"),
    **best_param_set
)

# model_trainer.fit(epochs=50, patience=200, batch_size=32)
model_trainer.load_checkpoint()

# Example: Plot SOFA Scores

In [ ]:
test_encoded = model_trainer.encode(split='test')

In [ ]:
# take a random sample to plot
sample_indexes = np.random.choice(len(test_encoded), size=50000)
encodings_mapped = TSNE(init='pca', verbose=True, metric='cosine').fit_transform(test_encoded[sample_indexes])

In [ ]:
severity_scores_test = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "test.csv"), index_col=0).iloc[:,-1].values
severity_scores_test = severity_scores_test[test_mask]

In [ ]:
plt.figure(figsize=(12, 6), dpi=300)
plt.subplot(121)
order = np.argsort(severity_scores_test[sample_indexes])
plt.scatter(encodings_mapped[order,0], encodings_mapped[order,1], s=2, lw=0, c=severity_scores_test[order], cmap='magma', vmax=20)
plt.xlabel("TSNE1")
plt.ylabel("TSNE2")
plt.gca().set_aspect(1.0)
plt.colorbar()
plt.subplot(122)
plt.hexbin(encodings_mapped[:,0], encodings_mapped[:,1], C=severity_scores_test[sample_indexes], lw=0, cmap='magma', vmax=20)
plt.xlabel("TSNE1")
plt.ylabel("TSNE2")
plt.gca().set_aspect(1.0)
plt.colorbar()
plt.suptitle("SOFA Score")
plt.tight_layout()
plt.show()

# Case Selection


faiss is required to run (`conda install -c pytorch faiss-cpu=1.9.0`)

In [ ]:
severity_scores_train = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "train.csv"), index_col=0).iloc[:,-1].values
severity_scores_train = severity_scores_train[train_mask]
severity_scores_val = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "val.csv"), index_col=0).iloc[:,-1].values
severity_scores_val = severity_scores_val[val_mask]
severity_scores_test = pd.read_csv(os.path.join(data_dir, "outcomes", "sofa", "test.csv"), index_col=0).iloc[:,-1].values
severity_scores_test = severity_scores_test[test_mask]

severity_cutoffs = np.quantile(severity_scores_train, np.linspace(0, 1, 9))
print("SOFA score cutoffs:", severity_cutoffs)
train_severity_quantiles = np.digitize(severity_scores_train, severity_cutoffs[:-1]) - 1
val_severity_quantiles = np.digitize(severity_scores_val, severity_cutoffs[:-1]) - 1
test_severity_quantiles = np.digitize(severity_scores_test, severity_cutoffs[:-1]) - 1

In [ ]:
def create_treatments(split):
    treatments = pd.concat([
        pd.read_csv(os.path.join(data_dir, "treatments", "fluids", f"{split}.csv"), index_col=0).drop(columns=["id", "intime"]),
        pd.read_csv(os.path.join(data_dir, "treatments", "vaso", f"{split}.csv"), index_col=0).drop(columns=["id", "intime"]),
    ], axis=1)
    treatments.columns = ["fluid", "vaso"]
    treatments['vaso'] = treatments['vaso'].fillna(0).astype(int)
    treatments['fluid'] = treatments['fluid'].fillna(0).astype(int)
    return treatments

treatments_train = create_treatments('train')[train_mask]
treatments_test = create_treatments('test')[test_mask]

In [ ]:
# example: nearest neighbors of the validation set instances in the training set
neighborhoods_dir = os.path.join(data_dir, "neighborhoods")
if not os.path.exists(neighborhoods_dir): os.mkdir(neighborhoods_dir)

if not os.path.exists(os.path.join(neighborhoods_dir, "train_neighbors.pkl")):
    train_encoded = model_trainer.encode(split='train')
    test_encoded = model_trainer.encode(split='test')
    train_neighb_train, train_neighb_dist_train = get_nearest_neighbors(train_encoded, 
                                                                    train_encoded, 
                                                                    train_df["id"].values, 
                                                                    reference_identical=True,
                                                                    stratify=(train_severity_quantiles, train_severity_quantiles))
    test_neighb_train, test_neighb_dist_train = get_nearest_neighbors(test_encoded, 
                                                                    train_encoded, 
                                                                    train_df["id"].values, 
                                                                    stratify=(test_severity_quantiles, train_severity_quantiles))

    with open(os.path.join(neighborhoods_dir, "train_neighbors.pkl"), "wb") as file:
        pickle.dump((train_neighb_train, train_neighb_dist_train), file)
    with open(os.path.join(neighborhoods_dir, "test_neighbors.pkl"), "wb") as file:
        pickle.dump((test_neighb_train, test_neighb_dist_train), file)
else:
    with open(os.path.join(neighborhoods_dir, "train_neighbors.pkl"), "rb") as file:
        train_neighb_train, train_neighb_dist_train = pickle.load(file)
    with open(os.path.join(neighborhoods_dir, "test_neighbors.pkl"), "rb") as file:
        test_neighb_train, test_neighb_dist_train = pickle.load(file)
    

In [ ]:
# Preprocess slicing variables
train_slicing_vars = pd.read_csv(os.path.join(data_dir, "slicing", "inputs_train.csv"), index_col=0)[train_mask].reset_index(drop=True)
eval_slicing_vars = pd.read_csv(os.path.join(data_dir, "slicing", "inputs_test.csv"), index_col=0)[test_mask].reset_index(drop=True)
COLUMNS_TO_DROP = [
    "Gender"
]
train_slicing_vars = train_slicing_vars.drop(columns=COLUMNS_TO_DROP)
eval_slicing_vars = eval_slicing_vars.drop(columns=COLUMNS_TO_DROP)

for c in eval_slicing_vars.columns:
    try:
        if len(np.unique(eval_slicing_vars[c])) == 2 and (np.unique(eval_slicing_vars[c]) == np.array([0, 1])).all():
            print(c)
            eval_slicing_vars[c] = eval_slicing_vars[c].astype(int).replace({0: 'No', 1: 'Yes'})
            train_slicing_vars[c] = train_slicing_vars[c].astype(int).replace({0: 'No', 1: 'Yes'})
    except: pass

eval_slicing_vars = eval_slicing_vars.fillna('Missing')
train_slicing_vars = train_slicing_vars.fillna('Missing')

# Identify Cases with High Abnormality

In [ ]:
RANGES_BY_SYSTEM = {
    "Cardiac": {
        "Lactic Acid": { "min": 0.5, "max": 1.6 },
        "Troponin": { "max": 1 }, # not sure what these units are
    },
    "Coags": {
        "PTT": { "min": 60, "max": 70 },
        "PT": { "min": 11, "max": 13.5 },
        "INR": { "min": 0.8, "max": 1.1 },
    },
    "ABGs": {
        "Arterial pH": { "min": 7.35, "max": 7.45 },
        "Arterial BE": { "min": -4, "max": 2 },
        "PaO2": { "min": 75, "max": 100 },
        "PaCO2": { "min": 35, "max": 45 },
        "Bicarbonate": { "min": 20, "max": 32 },
    },
    "ID": {
        "WBC Count": { "min": 4.5, "max": 11 },
    },
    "Hgb": {
        "Hemoglobin": { "male": { "min": 14.0, "max": 17.5 },
                        "female": { "min": 12.3, "max": 15.3 }},
        "Hematocrit": { "male": { "min": 41, "max": 50 },
                        "female": { "min": 36, "max": 48 }},
    },
    "Platelets": {
        "Platelet Count": { "min": 130, "max": 400 },
    },
    "Liver": {
        "AST": { "female": { "min": 9, "max": 25 }, "male": { "min": 10, "max": 40 } },
        "ALT": { "female": { "min": 7, "max": 30 }, "male": { "min": 10, "max": 55 } },
        "Total Bili": { "min": -1, "max": 5 },
        "Direct Bili": { "min": -1, "max": 2 },
    },
    "Chemistries": {
        "Potassium": { "min": 3.4, "max": 5 },
        "Sodium": { "min": 135, "max": 145 },
        "Chloride": { "min": 95, "max": 108 },
        "Glucose": { "min": 110, "max": 180 },
        "Ionized Ca": { "min": 1.1, "max": 1.4 },
    },
    "Renal": {
        "BUN": { "min": 8, "max": 25 },
        "Creatinine": {
            "female": { "min": 0.6, "max": 1.8 },
            "male": { "min": 0.8, "max": 2.4 },
          },
    },
    "Neuro": {
        "GCS Eye Opening": { "min": 3, "max": 6 },
        "GCS Motor Response": { "min": 3, "max": 7 },
        "GCS Verbal Response": { "min": 3, "max": 6 },
        "RASS": { "min": -3, "max": 2 }
    }
}

def _assign_bin_values(col_data, bin_spec):
    lower = bin_spec.get("min", np.quantile(col_data, 0.25))
    upper = bin_spec.get("max", np.quantile(col_data, 0.75))
    assert lower < upper, f"{bin_spec}"
    return np.digitize(col_data, [lower, upper]) != 1

EXTREME_VALUE_BIN_NAMES = {"normal": 0, "abnormal": 1}
def extreme_value_binning(col_name, norm_range, col_data, gender_data):
    if "female" in norm_range:
        return np.where(gender_data, 
                        _assign_bin_values(col_data, norm_range["female"]),
                        _assign_bin_values(col_data, norm_range["male"])), EXTREME_VALUE_BIN_NAMES
    return _assign_bin_values(col_data, norm_range), EXTREME_VALUE_BIN_NAMES

def get_abnormal_flags(states):
    abnormal_flags = []
    abnormal_flags_system = {}
    for system, system_ranges in RANGES_BY_SYSTEM.items():
        system_flags = np.vstack([
            np.logical_and(extreme_value_binning(lab_name, range_info, states[lab_name], states['gender_female'])[0],
                           states[f'{lab_name} Present'])
            for lab_name, range_info in system_ranges.items()
        ]).T
        abnormal_flags.append(pd.DataFrame(system_flags, columns=list(system_ranges.keys())))
        abnormal_flags_system[system] = system_flags.any(axis=1)
    abnormal_flags = pd.concat(abnormal_flags, axis=1)
    abnormal_flags_system = pd.DataFrame(abnormal_flags_system)
    return abnormal_flags, abnormal_flags_system

raw_test_states = pd.read_csv(os.path.join(data_dir, "model_features", "inputs_test.csv"), index_col=0)[test_mask].reset_index(drop=True)
abnormal_flags, abnormal_flags_system = get_abnormal_flags(raw_test_states)
abnormal_flags.mean(axis=0)

In [ ]:
matching_notes = pd.read_csv("../../AI-Clinician-MIMICIV/data/intermediates/patient_matched_notes.csv")
matching_notes = pd.merge(test_df[['id', 'intime']], matching_notes.drop_duplicates('note_id'), left_on='id', right_on='icustayid', how='left')
note_contains_sepsis = matching_notes.groupby(['id', 'intime']).agg({'target': lambda g: g.str.contains(r'sepsis|septic', case=False, regex=True).any()}).reset_index()
del matching_notes

cases_to_select = np.logical_and(
    np.logical_and(
        # interesting_masks.sum(axis=0) >= 4,
        test_neighb_dist_train[:,-1] <= np.quantile(test_neighb_dist_train[:,-1], 0.25),
        abnormal_flags.sum(axis=1) >= 5
    ),
    # abnormal_flags.sum(axis=1) >= 5,
    note_contains_sepsis['target']
)
print(cases_to_select.sum(), len(test_df['id'][cases_to_select].unique()))

cases_to_select = test_df[['id', 'intime']][cases_to_select]
cases_to_select['id'] = cases_to_select['id'].astype(int)
cases_to_select.to_csv(os.path.join(data_dir, 'interesting_prediction_cases.csv'), index=False)